# Superstore Sales Analysis: Data Understanding & Cleaning

**Goal:** understand the Superstore dataset and prepare a clean, analysis-ready version for EDA, SQL and Power BI.

**Input:** `data/raw/train.csv` (original file, never edited)  
**Output:** `data/processed/superstore_clean.csv`

**Steps:** 1) load and inspect, 2) clean (dates, postal codes, duplicates, validation, text), 3) create features, 4) save.

## 1. Load and inspect the data

The file is read with `encoding="latin-1"` because it contains non-UTF-8 characters. `shape`, `info()` and `head()` give a first look at size, column types and example rows.

**What we learned:** 9,800 rows and 18 columns. One row = one product line inside an order, so an order can span several rows (count `Order ID` uniquely to count orders). There are only three numeric columns (`Row ID`, `Postal Code`, `Sales`): **this version has no Profit, Quantity or Discount**, so all analysis is sales-based. `Order Date` and `Ship Date` are stored as text.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/train.csv", encoding="latin-1")

print("Shape:", df.shape)
df.info()
df.head()

Shape: (9800, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   object 
 13  Product ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub-Category   9800 non-null   object 
 16  Product Name   9800 non-null   object 
 17  Sales          9800 non-null   flo

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


### Summary statistics

`describe()` shows the spread of the numeric columns. `Sales` is strongly right-skewed: median about 54 but mean about 231 and max about 22,638, so a few very large purchases pull the average up. We keep them (they are real purchases) and use the median or a log scale when needed. `Postal Code` statistics are meaningless because it is an identifier, not a quantity.

In [2]:
df.describe()

,Row ID,Postal Code,Sales
count,9800.000000,9789.000000,9800.000000
mean,4900.500000,55273.322403,230.769059
std,2829.160653,32041.223413,626.651875
min,1.000000,1040.000000,0.444000
25%,2450.750000,23223.000000,17.248000
50%,4900.500000,58103.000000,54.490000
75%,7350.250000,90008.000000,210.605000
max,9800.000000,99301.000000,22638.480000


### Missing values and duplicates

**Result:** only `Postal Code` has missing values (11 rows). `duplicated()` returns 0 here, but only because `Row ID` is unique and makes every row different. We check again without it in section 2.3.

In [3]:
print(df.isna().sum())
print("Duplicates:", df.duplicated().sum())

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64
Duplicates: 0


## 2. Cleaning

### 2.1 Convert dates to datetime

Dates are text in **day/month/year** format (for example `08/11/2017` = 8 November 2017). Giving pandas the explicit `format="%d/%m/%Y"` prevents it from guessing month/day and silently mixing up dates. Proper datetime columns are needed for year, month and shipping-time calculations.

In [4]:
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%d/%m/%Y")
df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  format="%d/%m/%Y")

### 2.2 Missing postal codes

First we investigate instead of filling blindly: which rows are missing a postal code?

**Result:** all 11 missing values belong to one place, **Burlington, Vermont**. Since it is a single city, the missing value can be filled with that city's ZIP code (05401).

Postal codes are identifiers, not numbers, so they are converted to 5-character text with `zfill(5)` (this keeps leading zeros such as `05401`).

In [5]:
missing = df[df["Postal Code"].isna()]
print(missing[["City", "State"]].drop_duplicates())

            City    State
2234  Burlington  Vermont


In [6]:
# All 11 missing rows are Burlington, Vermont -> ZIP 05401
df.loc[df["Postal Code"].isna(), "Postal Code"] = 5401
df["Postal Code"] = df["Postal Code"].astype(int).astype(str).str.zfill(5)

# Check: 0 missing and every code has 5 characters
print(df["Postal Code"].isna().sum())
print(df["Postal Code"].str.len().value_counts())

0
Postal Code
5    9800
Name: count, dtype: int64


### 2.3 Duplicates (ignoring Row ID)

`Row ID` is just a counter, so a real duplicate must be searched in all the other columns. We display the duplicated rows first, then drop them.

**Result:** 1 pair of identical rows (a Global Leather chair, same order, same sales). With no quantity column to prove it was a genuine two-unit order, it is treated as a duplicate entry and removed, keeping the first occurrence. The impact is negligible (1 of 9,800 rows). Expected shape after removal: (9799, 18).

In [7]:
cols = df.columns.drop("Row ID")
print(df[df.duplicated(subset=cols, keep=False)])   # look at the rows first

df = df.drop_duplicates(subset=cols, keep="first").reset_index(drop=True)
print(df.shape)   # expect (9799, 18)

      Row ID        Order ID Order Date  Ship Date       Ship Mode  \
3405    3406  US-2015-150119 2015-04-23 2015-04-27  Standard Class   
3406    3407  US-2015-150119 2015-04-23 2015-04-27  Standard Class   

     Customer ID   Customer Name      Segment        Country      City State  \
3405    LB-16795  Laurel Beltran  Home Office  United States  Columbus  Ohio   
3406    LB-16795  Laurel Beltran  Home Office  United States  Columbus  Ohio   

     Postal Code Region       Product ID   Category Sub-Category  \
3405       43229   East  FUR-CH-10002965  Furniture       Chairs   
3406       43229   East  FUR-CH-10002965  Furniture       Chairs   

                                           Product Name    Sales  
3405  Global Leather Highback Executive Chair with P...  281.372  
3406  Global Leather Highback Executive Chair with P...  281.372  
(9799, 18)


### 2.4 Logical checks and category quality

- **Ship before order:** a ship date earlier than the order date would be impossible.
- **Non-positive sales:** sales should always be greater than 0.
- **Category values:** printing the unique values reveals typos or inconsistent spelling (for example `Consumer` vs `consumer`).

**Result:** both logical checks return 0, and every categorical column has clean, consistent values (4 ship modes, 3 segments, 4 regions, 3 categories, 17 sub-categories). No fixes needed.

In [8]:
print("Ship before order:", (df["Ship Date"] < df["Order Date"]).sum())
print("Non-positive sales:", (df["Sales"] <= 0).sum())
for col in ["Ship Mode", "Segment", "Region", "Category", "Sub-Category"]:
    print(col, df[col].unique())

Ship before order: 0
Non-positive sales: 0
Ship Mode ['Second Class' 'Standard Class' 'First Class' 'Same Day']
Segment ['Consumer' 'Corporate' 'Home Office']
Region ['South' 'West' 'Central' 'East']
Category ['Furniture' 'Office Supplies' 'Technology']
Sub-Category ['Bookcases' 'Chairs' 'Labels' 'Tables' 'Storage' 'Furnishings' 'Art'
 'Phones' 'Binders' 'Appliances' 'Paper' 'Accessories' 'Envelopes'
 'Fasteners' 'Supplies' 'Machines' 'Copiers']


### 2.5 Remove extra spaces in text columns

Leading or trailing spaces make identical values look different, which would split groups in SQL and Power BI. `str.strip()` removes them from the free-text columns.

In [9]:
df[["Customer Name", "City", "State", "Product Name"]] = (
    df[["Customer Name", "City", "State", "Product Name"]].apply(lambda s: s.str.strip())
)

## 3. Feature engineering

New columns derived from the dates make time-based analysis easy later: `Year`, `Month`, `Month Name`, `Quarter` (seasonality, growth) and `Shipping Days` = ship date minus order date (delivery performance).

In [10]:
df["Year"]  = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.month_name()
df["Quarter"] = df["Order Date"].dt.quarter
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

## 4. Save the cleaned dataset

The cleaned data is saved to `data/processed/`, leaving the raw file untouched, so every later step (EDA, SQL, Power BI) uses the same clean source.

**Final result:** 9,799 rows and 23 columns (18 original + 5 new). Both date columns are `datetime64`, `Postal Code` is text, and `Shipping Days` ranges from 0 to 7 days.

In [11]:
df.to_csv("../data/processed/superstore_clean.csv", index=False)

print("Final shape:", df.shape)
print(df.dtypes)
print(df["Shipping Days"].describe())

Final shape: (9799, 23)
Row ID                    int64
Order ID                 object
Order Date       datetime64[ns]
Ship Date        datetime64[ns]
Ship Mode                object
Customer ID              object
Customer Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal Code              object
Region                   object
Product ID               object
Category                 object
Sub-Category             object
Product Name             object
Sales                   float64
Year                      int32
Month                     int32
Month Name               object
Quarter                   int32
Shipping Days             int64
dtype: object
count    9799.000000
mean        3.961118
std         1.749703
min         0.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         7.000000
Name: Shipping Days, dtype: float64


## Summary of cleaning decisions

| Issue | Action |
|---|---|
| Dates stored as text (day/month/year) | Converted with an explicit format |
| 11 missing postal codes (all Burlington, VT) | Filled with ZIP 05401 |
| Postal code stored as a float | Converted to a 5-character string |
| 1 duplicate row (ignoring Row ID) | Removed |
| Logical errors and category typos | None found |
| Extra spaces in text columns | Stripped |
| Missing time fields | Added Year, Month, Month Name, Quarter, Shipping Days |